In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

JAR_DIR = "/home/iamaral/Documents/dev-projects/pipeline-kafka/jar"

spark = SparkSession.builder \
    .appName("SparkSQL-Kafka-Postgres") \
    .config("spark.jars", ",".join([
        f"{JAR_DIR}/postgresql-42.7.1.jar",
        f"{JAR_DIR}/spark-sql-kafka-0-10_2.13-4.1.2.jar",
        f"{JAR_DIR}/spark-token-provider-kafka-0-10_2.13-4.1.2.jar",
        f"{JAR_DIR}/kafka-clients-3.9.1.jar",
        f"{JAR_DIR}/commons-pool2-2.12.1.jar",
    ])) \
    .getOrCreate()

In [ ]:
df_pg = spark.read \
    .format("jdbc") \
    .option("driver", "org.postgresql.Driver") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "(SELECT \"CompraManha\", \"VendaManha\", \"PUCompraManha\", \"PUVendaManha\", \"PUBaseManha\", \"Data_Vencimento\", \"Data_Base\", \"Tipo\", dt_update FROM public.dadostesouroipca) AS dados") \
    .option("user", "postgres") \
    .option("password", "postgres") \
    .load()

df_pg.show()

df_pg.createOrReplaceTempView("tabela_ipca")


+-----------+----------+-------------+------------+-----------+---------------+----------+----+--------------------+
|CompraManha|VendaManha|PUCompraManha|PUVendaManha|PUBaseManha|Data_Vencimento| Data_Base|Tipo|           dt_update|
+-----------+----------+-------------+------------+-----------+---------------+----------+----+--------------------+
|       6.29|      6.37|       601.15|      593.62|     593.44|     2024-08-15|2007-10-25|IPCA|2026-06-20 08:01:...|
|       7.19|      7.25|       989.75|      985.59|     985.25|     2015-05-15|2007-10-25|IPCA|2026-06-20 08:01:...|
|        6.3|      6.38|       600.11|       592.6|      592.4|     2024-08-15|2007-10-24|IPCA|2026-06-20 08:01:...|
|       7.12|      7.18|       994.45|      990.27|     989.91|     2015-05-15|2007-10-24|IPCA|2026-06-20 08:01:...|
|       6.38|      6.46|       589.33|      581.93|     581.64|     2024-08-15|2007-09-28|IPCA|2026-06-20 08:01:...|
|       6.79|      6.85|      1011.96|     1007.65|    1007.14| 

In [ ]:
spark.sql("""
    SELECT Tipo, COUNT(*) AS total
    FROM tabela_ipca
    GROUP BY Tipo
""").show()


+----+-----+
|Tipo|total|
+----+-----+
|IPCA|55230|
+----+-----+



In [ ]:
df_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "postgres-dadostesouroipca") \
    .load()

# Convertendo o valor em string
df_kafka_str = df_kafka.selectExpr("CAST(value AS STRING)")
df_kafka_str.show()
df_kafka_str.createOrReplaceTempView("mensagens_kafka")

+--------------------+
|               value|
+--------------------+
|    \���(\!@{...|
|    �p=\nף!@��...|
|    R���Q!@q=...|
|    �p=\nף!@��...|
|    )\����!@R�...|
|    �����L!@�Q...|
|    ףp=\nW!@�(...|
|    )\����!@R�...|
|    �����L!@�Q...|
|    ��Q��!@H�...|
|    �����L!@�Q...|
|    ��Q��!@H�...|
|    ףp=\nW!@�(...|
|    )\����!@R�...|
|    R���Q!@q=...|
|    33333�!@\�...|
|    q=\nףp!@��...|
|    ��(\��!@�Q...|
|    ���Q�!@�z...|
|    ffffff!@��...|
+--------------------+
only showing top 20 rows


In [ ]:
spark.sql("""
    SELECT value, LENGTH(value) as tamanho
    FROM mensagens_kafka
""").show()


+--------------------+-------+
|               value|tamanho|
+--------------------+-------+
|    \���(\!@{...|     64|
|    �p=\nף!@��...|     61|
|    R���Q!@q=...|     63|
|    �p=\nף!@��...|     57|
|    )\����!@R�...|     64|
|    �����L!@�Q...|     63|
|    ףp=\nW!@�(...|     63|
|    )\����!@R�...|     58|
|    �����L!@�Q...|     61|
|    ��Q��!@H�...|     61|
|    �����L!@�Q...|     62|
|    ��Q��!@H�...|     63|
|    ףp=\nW!@�(...|     63|
|    )\����!@R�...|     63|
|    R���Q!@q=...|     65|
|    33333�!@\�...|     68|
|    q=\nףp!@��...|     64|
|    ��(\��!@�Q...|     64|
|    ���Q�!@�z...|     62|
|    ffffff!@��...|     65|
+--------------------+-------+
only showing top 20 rows


In [ ]:
df_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "postgres-dadostesouroipca") \
    .load()

# Inspeciona 1 registro bruto
amostra = df_kafka.selectExpr("CAST(value AS STRING) as value").limit(1).collect()
print(amostra[0]["value"])

    \���(\!@{�G�z!@{�G�P�@���(\3�@fffff0�@���ͪS�����AIPCAċ��g


In [ ]:
spark.stop()